Извлечение текстов с PERSON из NERUS

In [ ]:
!pip install nerus

In [ ]:
!ls

In [ ]:
from nerus import load_nerus

docs = load_nerus("nerus_lenta.conllu.gz")

doc = next(docs)

sent = doc.sents[0]

sent.text

In [ ]:
for token in sent.tokens:
    print(token.text, token.tag)

In [ ]:
def convert_tag(tag):
    if tag.endswith("PER"):
        return tag.replace("PER", "PERSON")
    return "O"

for token in sent.tokens:
    print(token.text, convert_tag(token.tag))

In [ ]:
from nerus import load_nerus

docs = load_nerus("nerus_lenta.conllu.gz")

dataset = []

for doc in docs:
    for sent in doc.sents:
        tokens = sent.tokens

        has_person = any("PER" in token.tag for token in tokens)

        if has_person:
            sentence_data = []

            for token in tokens:
                tag = token.tag

                if tag.endswith("PER"):
                    tag = tag.replace("PER", "PERSON")
                else:
                    tag = "O"

                sentence_data.append((token.text, tag))

            dataset.append(sentence_data)

    if len(dataset) > 5000:
        break

In [ ]:
for token, tag in dataset[0]:
    print(token, tag)

In [ ]:
with open("person_dataset_nerus.txt", "w", encoding="utf-8") as f:
    for sent in dataset:
        for token, tag in sent:
            f.write(f"{token} {tag}\n")
        f.write("\n")

ADDRESS

In [ ]:
from nerus import load_nerus

docs = load_nerus("nerus_lenta.conllu.gz")

address_markers = {
    "адрес", "адресу", "адресе", "адресом",
    "проживает", "проживал", "проживала", "проживают",
    "живет", "живёт", "жил", "жила",
    "находится", "расположен", "расположена", "расположено",
    "по", "адресу:",
    "ул.", "улица", "ул",
    "дом", "д.", "д",
    "кв.", "квартира", "квартире",
    "г.", "город",
    "область", "район",
    "проспект", "пр.", "пр",
    "переулок", "пер.", "пер",
    "шоссе", "бульвар", "наб.", "набережная"
}

address_candidates = []

for doc in docs:
    for sent in doc.sents:
        token_texts = [t.text for t in sent.tokens]
        token_tags = [t.tag for t in sent.tokens]

        has_loc = any("LOC" in tag for tag in token_tags)
        if not has_loc:
            continue

        sent_lower = sent.text.lower()

        marker_found = any(marker in sent_lower for marker in address_markers)

        if marker_found:
            address_candidates.append(sent)

        if len(address_candidates) >= 100:
            break
    if len(address_candidates) >= 100:
        break

for i, sent in enumerate(address_candidates[:50], 1):
    print(f"\n[{i}] {sent.text}")

Nerus хорошо подходит для PERSON, но плохо подходит для ADDRESS в смысле персональных данных.

**CORUS**

В работе Corus рассматривается не как единый размеченный датасет, а как каталог и программный интерфейс для доступа к публичным русскоязычным корпусам. Он используется для подбора и загрузки сырых текстов, на основе которых далее формируются собственные выборки для разметки и генерации примеров персональных данных.

Часть публичных новостных корпусов (например, ODS Interfax) оказалась недоступной или представленной в сокращённом виде, что потребовало использования альтернативных источников данных.

**Wikipedia**

In [ ]:
!wget https://dumps.wikimedia.org/ruwiki/latest/ruwiki-latest-pages-articles.xml.bz2

In [ ]:
!bzip2 -dk ruwiki-latest-pages-articles.xml.bz2

In [ ]:
from corus import load_wiki

records = load_wiki('ruwiki-latest-pages-articles.xml')

In [ ]:
from corus import load_wiki

records = load_wiki('ruwiki-latest-pages-articles.xml.bz2')
first = next(records)

print(first)
print(first.title)
print(first.text[:1000])

In [ ]:
import re

def clean_wiki_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r'==.*?==', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc
from razdel import sentenize

segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

In [ ]:
def extract_entities(text: str):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    entities = []
    for span in doc.spans:
        entities.append({
            'text': span.text,
            'type': span.type,
            'start': span.start,
            'stop': span.stop
        })
    return entities

In [ ]:
from corus import load_wiki
from razdel import sentenize

records = load_wiki('ruwiki-latest-pages-articles.xml.bz2')

person_sentences = []
negative_sentences = []

max_articles = 1000

for i, record in enumerate(records):
    if i >= max_articles:
        break

    text = clean_wiki_text(record.text)

    for sent in sentenize(text):
        sent_text = sent.text.strip()

        if len(sent_text) < 30 or len(sent_text) > 220:
            continue

        entities = extract_entities(sent_text)
        has_person = any(e['type'] == 'PER' for e in entities)

        if has_person:
            person_sentences.append({
                'title': record.title,
                'sentence': sent_text,
                'entities': entities
            })
        else:
            negative_sentences.append({
                'title': record.title,
                'sentence': sent_text
            })

print("PERSON-предложений:", len(person_sentences))

In [ ]:
for x in person_sentences[70:100]:
    print("TITLE:", x['title'])
    print("SENTENCE:", x['sentence'])
    print("ENTITIES:", x['entities'])
    print("---")

In [ ]:
import re
import pandas as pd

df = pd.DataFrame(person_sentences)
df.head()

In [ ]:
def get_per_entities(entities):
    return [e for e in entities if e['type'] == 'PER']

df['per_entities'] = df['entities'].apply(get_per_entities)
df['n_per'] = df['per_entities'].apply(len)
df['sent_len'] = df['sentence'].apply(len)
df['n_words'] = df['sentence'].apply(lambda x: len(x.split()))

In [ ]:
df = df[(df['sent_len'] >= 25) & (df['sent_len'] <= 220)].copy()

In [ ]:
df = df[df['n_per'] <= 5].copy()

In [ ]:
def is_good_person_text(text):
    text = text.strip()

    if len(text) < 2:
        return False

    if '(' in text or ')' in text or '*' in text or '=' in text:
        return False

    if re.search(r'\d', text):
        return False

    if len(text.split()) > 5:
        return False

    return True

In [ ]:
def filter_good_pers(entities):
    return [e for e in entities if is_good_person_text(e['text'])]

df['good_per_entities'] = df['per_entities'].apply(filter_good_pers)
df['n_good_per'] = df['good_per_entities'].apply(len)

df = df[df['n_good_per'] > 0].copy()

In [ ]:
df = df.drop_duplicates(subset=['sentence']).copy()

In [ ]:
df['title_count'] = df.groupby('title')['title'].transform('count')
df['title'].value_counts().head(20)

In [ ]:
df = (
    df.groupby('title', group_keys=False)
      .head(30)
      .copy()
)

In [ ]:
def person_format(person_text):
    words = person_text.split()

    if re.search(r'^(?:[А-ЯЁ]\.\s*){1,3}[А-ЯЁ][а-яё\-]+$', person_text):
        return 'initials_surname'

    if re.search(r'^(?:[А-ЯЁ]\.\s*){2,3}$', person_text):
        return 'initials_only'

    if len(words) == 1:
        return 'one_word'

    if len(words) == 2:
        return 'two_words'

    if len(words) >= 3:
        return 'three_plus_words'

    return 'other'

In [ ]:
df['main_person'] = df['good_per_entities'].apply(lambda x: x[0]['text'])
df['person_format'] = df['main_person'].apply(person_format)

In [ ]:
def normalize_person_name(name):
    name = name.lower().strip()
    name = re.sub(r'[^\w\s\-ё]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

df['person_norm'] = df['main_person'].apply(normalize_person_name)

In [ ]:
df = (
    df.groupby('person_norm', group_keys=False)
      .head(2)
      .copy()
)

In [ ]:
def length_bucket(n):
    if n < 70:
        return 'short'
    elif n < 130:
        return 'medium'
    else:
        return 'long'

df['length_bucket'] = df['sent_len'].apply(length_bucket)

In [ ]:
def per_count_bucket(n):
    if n == 1:
        return 'one_per'
    elif n == 2:
        return 'two_per'
    else:
        return 'three_plus_per'

df['per_count_bucket'] = df['n_good_per'].apply(per_count_bucket)

In [ ]:
print(df['person_format'].value_counts())
print(df['length_bucket'].value_counts())
print(df['per_count_bucket'].value_counts())

In [ ]:
sample_parts = []

for length_group, n_take in [('short', 150), ('medium', 200), ('long', 150)]:
    part = df[df['length_bucket'] == length_group].copy()

    for fmt, fmt_take in [
        ('one_word', n_take // 5),
        ('two_words', n_take // 3),
        ('three_plus_words', n_take // 5),
        ('initials_surname', n_take // 5),
        ('other', n_take // 10),
    ]:
        chunk = part[part['person_format'] == fmt].head(fmt_take)
        sample_parts.append(chunk)

In [ ]:
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.drop_duplicates(subset=['sentence']).copy()
print(len(sample_df))

In [ ]:
import pandas as pd

sample_df = pd.concat(sample_parts, ignore_index=True)

sample_df = sample_df.drop_duplicates(subset=['sentence']).copy()

print("После объединения:", len(sample_df))

In [ ]:
sample_df.to_csv('wiki_person_sample_500.csv', index=False, encoding='utf-8-sig')

In [ ]:
from google.colab import files
files.download('wiki_person_sample_500.csv')

**Taiga**

In [ ]:
from corus import load_taiga_social

In [ ]:
path = 'data/taiga/social.tar.gz'

In [ ]:
from corus import load_taiga_social

social_path = 'data/taiga/social.tar.gz'
records = load_taiga_social(social_path)

first = next(records)
print(first)

In [ ]:
print(type(first))
print(first._fields if hasattr(first, '_fields') else dir(first))

In [ ]:
import re
import pandas as pd
from collections import defaultdict

from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc
from razdel import sentenize

segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

In [ ]:
def get_record_text(record):
    if hasattr(record, '_asdict'):
        rec = record._asdict()
    else:
        rec = {attr: getattr(record, attr) for attr in dir(record)
               if not attr.startswith('_') and not callable(getattr(record, attr))}

    for key in ['text', 'content', 'body', 'message']:
        if key in rec and rec[key]:
            return str(rec[key])

    for key in rec:
        if isinstance(rec[key], str) and len(rec[key]) > 20:
            return rec[key]

    return ""

In [ ]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace('\n', ' ')
    text = text.replace('\\n', ' ')

    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
def extract_person_spans(text: str):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    persons = []
    for span in doc.spans:
        if span.type == 'PER':
            persons.append({
                'text': span.text,
                'start': span.start,
                'stop': span.stop
            })
    return persons

In [ ]:
def person_format(person_text: str) -> str:
    words = person_text.split()

    if re.search(r'^(?:[А-ЯЁA-Z]\.\s*){1,3}[А-ЯЁA-Z][а-яёa-z\-]+$', person_text):
        return 'initials_surname'

    if len(words) == 1:
        return 'one_word'

    if len(words) == 2:
        return 'two_words'

    if len(words) >= 3:
        return 'three_plus_words'

    return 'other'

In [ ]:
def normalize_person_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r'[^\w\s\-ё]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

In [ ]:
def length_bucket(n_chars: int) -> str:
    if n_chars < 70:
        return 'short'
    elif n_chars < 130:
        return 'medium'
    return 'long'

In [ ]:
records = load_taiga_social(social_path)

person_candidates = []
seen_sentences = set()

max_records = 3000

for i, record in enumerate(records):
    if i >= max_records:
        break

    raw_text = get_record_text(record)
    text = clean_text(raw_text)

    if not text:
        continue

    for sent in sentenize(text):
        sent_text = sent.text.strip()

        if len(sent_text) < 20 or len(sent_text) > 220:
            continue

        n_words = len(sent_text.split())
        if n_words < 3 or n_words > 35:
            continue

        persons = extract_person_spans(sent_text)
        if not persons:
            continue

        good_persons = []
        for p in persons:
            ptext = p['text'].strip()
            if len(ptext) < 2:
                continue
            if len(ptext.split()) > 5:
                continue
            if re.search(r'\d', ptext):
                continue
            good_persons.append(p)

        if not good_persons:
            continue

        if sent_text in seen_sentences:
            continue
        seen_sentences.add(sent_text)

        main_person = good_persons[0]['text']

        person_candidates.append({
            'sentence': sent_text,
            'persons': [p['text'] for p in good_persons],
            'main_person': main_person,
            'person_norm': normalize_person_name(main_person),
            'person_format': person_format(main_person),
            'n_persons': len(good_persons),
            'n_words': n_words,
            'length_chars': len(sent_text),
            'length_bucket': length_bucket(len(sent_text))
        })

print("Найдено кандидатов:", len(person_candidates))

In [ ]:
for x in person_candidates[:15]:
    print(x['sentence'])
    print("PERSON:", x['persons'])
    print("FORMAT:", x['person_format'], "| LEN:", x['length_bucket'], "| WORDS:", x['n_words'])
    print("---")

**Winer**

In [ ]:
!wget https://github.com/dice-group/FOX/raw/master/input/Wikiner/aij-wikiner-ru-wp3.bz2


In [ ]:
!bunzip2 -f aij-wikiner-ru-wp3.bz2


In [ ]:
!ls -lh

In [ ]:
import re
import pandas as pd
from collections import defaultdict

In [ ]:
def parse_wikiner_token(tok):
    parts = tok.split('|')
    if len(parts) < 3:
        return None

    word = parts[0]
    pos = parts[-2]
    tag = parts[-1]
    return word, pos, tag

In [ ]:
with open('aij-wikiner-ru-wp3', encoding='utf-8', errors='ignore') as f:
    for i in range(5):
        line = f.readline().strip()
        print(line[:500])
        print('---')

In [ ]:
sentences = []

with open('aij-wikiner-ru-wp3', encoding='utf-8', errors='ignore') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        raw_tokens = line.split()
        parsed = []

        for tok in raw_tokens:
            item = parse_wikiner_token(tok)
            if item is not None:
                parsed.append(item)

        if not parsed:
            continue

        words = [w for w, pos, tag in parsed]
        tags = [tag for w, pos, tag in parsed]

        sentence_text = ' '.join(words)

        sentences.append({
            'sentence': sentence_text,
            'tokens': words,
            'tags': tags
        })

print("Всего предложений:", len(sentences))
print(sentences[2]['sentence'][:300])
print(sentences[2]['tags'][:20])

In [ ]:
person_sentences = []

for item in sentences:
    tags = item['tags']
    if any('PER' in tag for tag in tags):
        person_sentences.append(item)

print("Предложений с PERSON:", len(person_sentences))

In [ ]:
def extract_person_entities(tokens, tags):
    entities = []
    current = []

    for tok, tag in zip(tokens, tags):
        if tag == 'I-PER':
            current.append(tok)
        else:
            if current:
                entities.append(' '.join(current))
                current = []

    if current:
        entities.append(' '.join(current))

    return entities

In [ ]:
for x in person_sentences[:10]:
    ents = extract_person_entities(x['tokens'], x['tags'])
    print(x['sentence'])
    print("PERSON:", ents)
    print("---")

In [ ]:
def is_bad_person(person_text: str) -> bool:

    roman_pattern = r'\b[IVXLCDM]+\b'

    if re.search(roman_pattern, person_text):
        return True

    return False

In [ ]:
rows = []

for item in person_sentences:
    persons = extract_person_entities(item['tokens'], item['tags'])

    persons = [p for p in persons if not is_bad_person(p)]

    if not persons:
        continue

    sentence = item['sentence']

    rows.append({
        'sentence': sentence,
        'tokens': item['tokens'],
        'tags': item['tags'],
        'persons': persons,
        'main_person': persons[0],
        'n_persons': len(persons),
        'length_chars': len(sentence),
        'n_words': len(item['tokens'])
    })

df = pd.DataFrame(rows)
print(df.head())
print("Всего PERSON-кандидатов после исправления:", len(df))

In [ ]:
wiki_df = pd.read_csv('wiki_person_sample_500.csv')
print(wiki_df.head())

In [ ]:
import re

def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[«»"“”„]', '', text)
    return text

def normalize_person_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r'[^\w\s\-ё]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

In [ ]:
wiki_sentence_set = set(wiki_df['sentence'].astype(str).apply(normalize_text))
wiki_person_set = set(wiki_df['main_person'].astype(str).apply(normalize_person_name))


In [ ]:
def person_format(person_text: str) -> str:
    words = person_text.split()

    if re.search(r'^(?:[А-ЯЁA-Z]\.\s*){1,3}[А-ЯЁA-Z][а-яёa-z\-]+$', person_text):
        return 'initials_surname'
    if len(words) == 1:
        return 'one_word'
    if len(words) == 2:
        return 'two_words'
    if len(words) >= 3:
        return 'three_plus_words'
    return 'other'

def length_bucket(n_chars: int) -> str:
    if n_chars < 70:
        return 'short'
    elif n_chars < 130:
        return 'medium'
    return 'long'

df['sentence_norm'] = df['sentence'].apply(normalize_text)
df['person_norm'] = df['main_person'].apply(normalize_person_name)
df['person_format'] = df['main_person'].apply(person_format)
df['length_bucket'] = df['length_chars'].apply(length_bucket)

In [ ]:
df_unique = df[
    (~df['sentence_norm'].isin(wiki_sentence_set)) &
    (~df['person_norm'].isin(wiki_person_set))
].copy()


In [ ]:
df_unique = df_unique[
    (df_unique['length_chars'] >= 25) &
    (df_unique['length_chars'] <= 220) &
    (df_unique['n_words'] >= 4) &
    (df_unique['n_words'] <= 35) &
    (df_unique['n_persons'] >= 1) &
    (df_unique['n_persons'] <= 5)
].copy()

df_unique = df_unique.drop_duplicates(subset=['sentence']).copy()

df_unique = (
    df_unique.groupby('person_norm', group_keys=False)
             .head(2)
             .copy()
)

print(df_unique['person_format'].value_counts())
print(df_unique['length_bucket'].value_counts())

In [ ]:
sample_parts = []

for length_group, n_take in [('short', 40), ('medium', 100), ('long', 60)]:
    part = df_unique[df_unique['length_bucket'] == length_group].copy()

    quotas = [
        ('one_word', n_take // 4),
        ('two_words', n_take // 3),
        ('three_plus_words', n_take // 5),
        ('initials_surname', n_take // 5),
        ('other', n_take // 10),
    ]

    for fmt, fmt_take in quotas:
        chunk = part[part['person_format'] == fmt].head(fmt_take)
        sample_parts.append(chunk)

In [ ]:
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.drop_duplicates(subset=['sentence']).copy()


In [ ]:
print(sample_df[['sentence', 'main_person', 'persons', 'person_format', 'length_bucket']].head(50))

In [ ]:
sample_df.to_csv('winer_person_sample_200_no_wiki_overlap.csv', index=False, encoding='utf-8-sig')

In [ ]:
from google.colab import files

files.download('winer_person_sample_200_no_wiki_overlap.csv')

**Mokoron Russian Twitter Corpus**

In [ ]:
!pip install corus natasha razdel pandas -q

In [ ]:
import re
import pandas as pd
from collections import defaultdict

from corus import load_mokoron
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc
from razdel import sentenize

In [ ]:
segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

In [ ]:
!wget "https://www.dropbox.com/s/9egqjszeicki4ho/db.sql?dl=1" -O db.sql

In [ ]:
records = load_mokoron('db.sql')

first = next(records)
print(first)

In [ ]:
def get_record_text(record):
    if hasattr(record, '_asdict'):
        rec = record._asdict()
    else:
        rec = {
            attr: getattr(record, attr)
            for attr in dir(record)
            if not attr.startswith('_') and not callable(getattr(record, attr))
        }

    for key in ['text', 'content', 'body', 'tweet']:
        if key in rec and rec[key]:
            return str(rec[key])

    for key, value in rec.items():
        if isinstance(value, str) and len(value) > 10:
            return value

    return ""

In [ ]:
records = load_mokoron('db.sql')
for i, record in enumerate(records):
    text = get_record_text(record)
    print("TEXT:", text[:300])
    print("---")
    if i >= 4:
        break

In [ ]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace('\n', ' ')
    text = text.replace('\\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def extract_person_spans(text: str):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    persons = []
    for span in doc.spans:
        if span.type == 'PER':
            persons.append({
                'text': span.text,
                'start': span.start,
                'stop': span.stop
            })
    return persons

In [ ]:
def is_bad_sentence(sent_text: str) -> bool:
    s = sent_text.strip()

    if len(s) < 20 or len(s) > 220:
        return True

    if re.search(r'https?://\S+|www\.\S+', s):
        return True

    if '@' in s:
        return True

    if '#' in s:
        return True

    if re.search(r'[!?]{2,}|\.{3,}', s):
        return True

    if re.search(r'[:;=8xX]-?[)(DPp/\\]+', s):
        return True

    return False

In [ ]:
def is_too_noisy(sent_text: str) -> bool:
    tokens = sent_text.split()
    if not tokens:
        return True

    short_ratio = sum(1 for t in tokens if len(t) <= 2) / len(tokens)

    upper_ratio = sum(1 for t in tokens if re.fullmatch(r'[А-ЯЁA-Z]{2,}', t)) / len(tokens)

    latin_ratio = sum(1 for t in tokens if re.search(r'[A-Za-z]', t)) / len(tokens)

    digit_ratio = sum(1 for t in tokens if re.search(r'\d', t)) / len(tokens)

    if short_ratio > 0.45:
        return True

    if upper_ratio > 0.25:
        return True

    if latin_ratio > 0.35:
        return True

    if digit_ratio > 0.30:
        return True

    return False

In [ ]:
def is_bad_person(person_text: str) -> bool:
    p = person_text.strip()

    if len(p) < 2:
        return True

    if re.search(r'\b[IVXLCDM]+\b', p):
        return True

    if '(' in p or ')' in p or '*' in p or re.search(r'\d', p):
        return True

    if len(p.split()) > 5:
        return True

    return False

In [ ]:
def person_format(person_text: str) -> str:
    words = person_text.split()

    if re.search(r'^(?:[А-ЯЁA-Z]\.\s*){1,3}[А-ЯЁA-Z][а-яёa-z\-]+$', person_text):
        return 'initials_surname'
    if len(words) == 1:
        return 'one_word'
    if len(words) == 2:
        return 'two_words'
    if len(words) >= 3:
        return 'three_plus_words'
    return 'other'

def normalize_person_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r'[^\w\s\-ё]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

def length_bucket(n_chars: int) -> str:
    if n_chars < 70:
        return 'short'
    elif n_chars < 130:
        return 'medium'
    return 'long'

In [ ]:
records = load_mokoron('db.sql')

person_candidates = []
seen_sentences = set()

max_records = 50000
target_raw_candidates = 1200

for i, record in enumerate(records):
    if i >= max_records:
        break

    text = clean_text(get_record_text(record))
    if not text:
        continue

    for sent in sentenize(text):
        sent_text = sent.text.strip()

        if is_bad_sentence(sent_text):
            continue

        if is_too_noisy(sent_text):
            continue

        n_words = len(sent_text.split())
        if n_words < 3 or n_words > 30:
            continue

        persons = extract_person_spans(sent_text)
        if not persons:
            continue

        good_persons = [p for p in persons if not is_bad_person(p['text'])]
        if not good_persons:
            continue

        if sent_text in seen_sentences:
            continue
        seen_sentences.add(sent_text)

        main_person = good_persons[0]['text']

        person_candidates.append({
            'sentence': sent_text,
            'persons': [p['text'] for p in good_persons],
            'main_person': main_person,
            'person_norm': normalize_person_name(main_person),
            'person_format': person_format(main_person),
            'n_persons': len(good_persons),
            'n_words': n_words,
            'length_chars': len(sent_text),
            'length_bucket': length_bucket(len(sent_text))
        })

        if len(person_candidates) >= target_raw_candidates:
            break

    if len(person_candidates) >= target_raw_candidates:
        break


In [ ]:
for x in person_candidates[:20]:
    print(x['sentence'])
    print("PERSON:", x['persons'])
    print("FORMAT:", x['person_format'], "| LEN:", x['length_bucket'], "| WORDS:", x['n_words'])
    print("---")

In [ ]:
df = pd.DataFrame(person_candidates)

In [ ]:
df = (
    df.groupby('person_norm', group_keys=False)
      .head(2)
      .copy()
)

print("После ограничения повторов по PERSON:", len(df))

In [ ]:
print(df['person_format'].value_counts())
print()
print(df['length_bucket'].value_counts())

In [ ]:
sample_parts = []

for length_group, n_take in [('short', 100), ('medium', 150), ('long', 150)]:
    part = df[df['length_bucket'] == length_group].copy()

    quotas = [
        ('one_word', n_take // 4),
        ('two_words', n_take // 3),
        ('three_plus_words', n_take // 5),
        ('initials_surname', n_take // 5),
        ('other', n_take // 10),
    ]

    for fmt, fmt_take in quotas:
        chunk = part[part['person_format'] == fmt].head(fmt_take)
        sample_parts.append(chunk)

In [ ]:
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.drop_duplicates(subset=['sentence']).copy()


In [ ]:
print(sample_df['person_format'].value_counts())
print()
print(sample_df['length_bucket'].value_counts())
print()
print(sample_df[['sentence', 'main_person', 'persons', 'person_format', 'length_bucket', 'n_words']].head(20))

In [ ]:
sample_df.to_csv('mokoron_person_sample_200.csv', index=False, encoding='utf-8-sig')

In [ ]:
from google.colab import files

files.download('mokoron_person_sample_200.csv')

**NEREL**

In [ ]:
!pip install razdel pandas -q

In [ ]:
!git clone https://github.com/nerel-ds/NEREL.git


In [ ]:
!find NEREL -maxdepth 3 | sed -n '1,200p'

In [ ]:
import os
import re
import pandas as pd
from razdel import sentenize

In [ ]:
pairs = []

for root, dirs, files in os.walk('NEREL'):
    txt_files = [f for f in files if f.endswith('.txt')]
    for txt_file in txt_files:
        txt_path = os.path.join(root, txt_file)
        ann_path = txt_path[:-4] + '.ann'
        if os.path.exists(ann_path):
            pairs.append((txt_path, ann_path))

print(pairs[:10])

In [ ]:
def parse_brat_entities(ann_path):
    entities = []

    with open(ann_path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith('T'):
                continue

            parts = line.split('\t')
            if len(parts) < 3:
                continue

            ent_id = parts[0]
            span_info = parts[1]
            ent_text = parts[2]

            span_parts = span_info.split()

            if len(span_parts) < 3:
                continue

            ent_type = span_parts[0]

            if ';' in span_info:
                continue

            try:
                start = int(span_parts[1])
                stop = int(span_parts[2])
            except ValueError:
                continue

            entities.append({
                'id': ent_id,
                'type': ent_type,
                'start': start,
                'stop': stop,
                'text': ent_text
            })

    return entities

In [ ]:
from collections import Counter

type_counter = Counter()

for txt_path, ann_path in pairs[:100]:
    ents = parse_brat_entities(ann_path)
    for e in ents:
        type_counter[e['type']] += 1

print(type_counter.most_common(50))

In [ ]:
PERSON_TYPES = {
    'PERSON',
    'PER',
    'PERSON_NAME'
}

In [ ]:
def get_person_sentences_from_doc(text, entities):
    person_entities = [e for e in entities if e['type'] in PERSON_TYPES]

    results = []

    for sent in sentenize(text):
        sent_text = sent.text.strip()
        sent_start = sent.start
        sent_stop = sent.stop

        inside = []
        for e in person_entities:
            if e['start'] >= sent_start and e['stop'] <= sent_stop:
                inside.append({
                    'text': e['text'],
                    'start': e['start'] - sent_start,
                    'stop': e['stop'] - sent_start
                })

        if inside:
            results.append({
                'sentence': sent_text,
                'persons': inside
            })

    return results

In [ ]:
def person_format(person_text: str) -> str:
    words = person_text.split()

    if re.search(r'^(?:[А-ЯЁA-Z]\.\s*){1,3}[А-ЯЁA-Z][а-яёa-z\-]+$', person_text):
        return 'initials_surname'

    if len(words) == 1:
        return 'one_word'
    if len(words) == 2:
        return 'two_words'
    if len(words) >= 3:
        return 'three_plus_words'
    return 'other'

In [ ]:
def normalize_person_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r'[^\w\s\-ё]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

In [ ]:
def length_bucket(n_chars: int) -> str:
    if n_chars < 70:
        return 'short'
    elif n_chars < 130:
        return 'medium'
    return 'long'

In [ ]:
def is_bad_person(person_text: str) -> bool:
    p = person_text.strip()

    if len(p) < 2:
        return True

    if re.search(r'\b[IVXLCDM]+\b', p):
        return True

    if '(' in p or ')' in p or '*' in p or re.search(r'\d', p):
        return True

    if len(p.split()) > 5:
        return True

    return False

In [ ]:
def is_bad_sentence(sent_text: str) -> bool:
    s = sent_text.strip()

    if len(s) < 25 or len(s) > 220:
        return True

    if s.count('=') > 0:
        return True

    return False

In [ ]:
rows = []
seen_sentences = set()

for i, (txt_path, ann_path) in enumerate(pairs):
    text = read_text_safe(txt_path)
    entities = parse_brat_entities(ann_path)

    sent_items = get_person_sentences_from_doc(text, entities)

    for item in sent_items:
        sentence = item['sentence']

        if is_bad_sentence(sentence):
            continue

        persons = [p['text'] for p in item['persons'] if not is_bad_person(p['text'])]
        persons = list(dict.fromkeys(persons))

        if not persons:
            continue

        if sentence in seen_sentences:
            continue
        seen_sentences.add(sentence)

        main_person = persons[0]

        rows.append({
            'source_txt': txt_path,
            'sentence': sentence,
            'persons': persons,
            'main_person': main_person,
            'person_norm': normalize_person_name(main_person),
            'person_format': person_format(main_person),
            'n_persons': len(persons),
            'length_chars': len(sentence),
            'n_words': len(sentence.split()),
            'length_bucket': length_bucket(len(sentence))
        })


In [ ]:
df = pd.DataFrame(rows)
print(df.head())
print("Всего:", len(df))

In [ ]:
df = df[
    (df['length_chars'] >= 25) &
    (df['length_chars'] <= 220) &
    (df['n_words'] >= 4) &
    (df['n_words'] <= 35) &
    (df['n_persons'] >= 1) &
    (df['n_persons'] <= 5)
].copy()

df = df.drop_duplicates(subset=['sentence']).copy()

df = (
    df.groupby('person_norm', group_keys=False)
      .head(2)
      .copy()
)

print(df['person_format'].value_counts())
print(df['length_bucket'].value_counts())

In [ ]:
sample_parts = []

for length_group, n_take in [('short', 60), ('medium', 80), ('long', 60)]:
    part = df[df['length_bucket'] == length_group].copy()

    quotas = [
        ('one_word', n_take // 4),
        ('two_words', n_take // 3),
        ('three_plus_words', n_take // 5),
        ('initials_surname', n_take // 5),
        ('other', n_take // 10),
    ]

    for fmt, fmt_take in quotas:
        chunk = part[part['person_format'] == fmt].head(fmt_take)
        sample_parts.append(chunk)

In [ ]:
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.drop_duplicates(subset=['sentence']).copy()


In [ ]:
remaining = df[~df['sentence'].isin(sample_df['sentence'])].copy()

need = 200 - len(sample_df)

if need > 0 and len(remaining) > 0:
    extra = remaining.sample(min(need, len(remaining)), random_state=42)
    sample_df = pd.concat([sample_df, extra], ignore_index=True)

sample_df = sample_df.drop_duplicates(subset=['sentence']).head(200).copy()


In [ ]:
print(sample_df['person_format'].value_counts())
print()
print(sample_df['length_bucket'].value_counts())
print()
print(sample_df[['sentence', 'main_person', 'persons', 'person_format', 'length_bucket', 'n_words']].head(20))

In [ ]:
sample_df.to_csv('nerel_person_sample_200.csv', index=False, encoding='utf-8-sig')

In [ ]:
from google.colab import files

files.download('nerel_person_sample_200.csv')

**FactRuEval-2016**

**Table of formats**

In [ ]:
!pip install datasets pandas -q

In [ ]:
import re
import pandas as pd
from datasets import load_dataset

In [ ]:
DATASET_NAME = "gusevski/factrueval2016"

dataset = load_dataset(DATASET_NAME)
dataset

In [ ]:
print(dataset)
print(dataset["train"])
print(dataset["train"][0].keys())

In [ ]:
all_items = []

for split_name in dataset.keys():
    split_data = dataset[split_name]

    for row in split_data:
        if "data" in row:
            for item in row["data"]:
                item_copy = dict(item)
                item_copy["split"] = split_name
                all_items.append(item_copy)

print(all_items[0])

In [ ]:
df = pd.DataFrame(all_items)

print(df.head())
print(df.columns)
print(len(df))

In [ ]:
label_names = [
    "O",
    "B-PER", "I-PER",
    "B-LOC", "I-LOC",
    "B-ORG", "I-ORG"
]

def decode_tags(tags):
    return [label_names[t] for t in tags]

In [ ]:
df["ner_tags_decoded"] = df["ner_tags"].apply(decode_tags)

print(df.iloc[0]["tokens"])
print(df.iloc[0]["ner_tags"])
print(df.iloc[0]["ner_tags_decoded"])

In [ ]:
def extract_person_entities(tokens, tags):
    entities = []
    current = []

    for tok, tag in zip(tokens, tags):
        if tag == "B-PER":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-PER":
            if current:
                current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
                current = []

    if current:
        entities.append(" ".join(current))

    return entities

In [ ]:
df["persons"] = df.apply(
    lambda row: extract_person_entities(row["tokens"], row["ner_tags_decoded"]),
    axis=1
)

df["n_persons"] = df["persons"].apply(len)
df["sentence"] = df["tokens"].apply(lambda x: " ".join(x))

df_person = df[df["n_persons"] > 0].copy()

print(df_person[["sentence", "persons"]].head(10))

In [ ]:
person_rows = []

for _, row in df_person.iterrows():
    sentence = row["sentence"]
    split_name = row["split"]

    for person in row["persons"]:
        person_rows.append({
            "split": split_name,
            "sentence": sentence,
            "person": person,
            "n_words_in_sentence": len(row["tokens"]),
            "sentence_len_chars": len(sentence)
        })

person_df = pd.DataFrame(person_rows)

print(person_df.head(10))

In [ ]:
def is_bad_person(person_text: str) -> bool:
    text = person_text.strip()

    if re.search(r"\b[IVXLCDM]+\b", text):
        return True

    if len(text) < 2:
        return True

    return False

In [ ]:
person_df = person_df[~person_df["person"].apply(is_bad_person)].copy()

print(len(person_df))

In [ ]:
def person_format(person_text: str) -> str:
    text = person_text.strip()
    words = text.split()

    if re.fullmatch(r'(?:[А-ЯЁA-Z]\.\s*){1,3}[А-ЯЁA-Z][а-яёa-z\-]+', text):
        return "initials_surname"

    if re.fullmatch(r'(?:[А-ЯЁA-Z]\.\s*){2,3}', text):
        return "initials_only"

    if len(words) == 1:
        return "one_word"

    if len(words) == 2:
        return "two_words"

    if len(words) >= 3:
        return "three_plus_words"

    return "other"

In [ ]:
def normalize_person(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s\-ё]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
person_df["person_format"] = person_df["person"].apply(person_format)
person_df["person_norm"] = person_df["person"].apply(normalize_person)
person_df["n_words_in_person"] = person_df["person"].apply(lambda x: len(x.split()))

In [ ]:
format_table = (
    person_df["person_format"]
    .value_counts(dropna=False)
    .rename_axis("person_format")
    .reset_index(name="count")
)

format_table["share"] = (format_table["count"] / format_table["count"].sum()).round(4)

format_table

In [ ]:
unique_person_df = person_df.drop_duplicates(subset=["person_norm"]).copy()

unique_format_table = (
    unique_person_df["person_format"]
    .value_counts(dropna=False)
    .rename_axis("person_format")
    .reset_index(name="unique_count")
)

unique_format_table["unique_share"] = (
    unique_format_table["unique_count"] / unique_format_table["unique_count"].sum()
).round(4)

unique_format_table

In [ ]:
final_format_table = format_table.merge(
    unique_format_table,
    on="person_format",
    how="outer"
).fillna(0)

final_format_table

In [ ]:
examples_by_format = (
    person_df.groupby("person_format")["person"]
    .apply(lambda x: list(pd.Series(x).drop_duplicates().head(10)))
    .reset_index(name="examples")
)

examples_by_format

In [ ]:
print("Уникальных PERSON:", person_df["person_norm"].nunique())
print()
print(final_format_table)

In [ ]:
person_df.to_csv("factrueval_person_mentions.csv", index=False, encoding="utf-8-sig")
final_format_table.to_csv("factrueval_person_format_table.csv", index=False, encoding="utf-8-sig")
examples_by_format.to_csv("factrueval_person_format_examples.csv", index=False, encoding="utf-8-sig")


In [ ]:
from google.colab import files

files.download("factrueval_person_format_table.csv")
files.download("factrueval_person_format_examples.csv")

In [ ]:
files.download("factrueval_person_mentions.csv")

**PERSON**

In [ ]:
print(person_df.columns)
print(person_df.head())

In [ ]:
target_formats = ["initials_surname", "three_plus_words", "two_words"]

df_sel = person_df[person_df["person_format"].isin(target_formats)].copy()

print("После отбора нужных форматов:", len(df_sel))
print(df_sel["person_format"].value_counts())

In [ ]:
df_sel = df_sel.drop_duplicates(subset=["sentence", "person"]).copy()

print("После удаления дублей sentence+person:", len(df_sel))

In [ ]:
df_sel = (
    df_sel.groupby("person_norm", group_keys=False)
          .head(2)
          .copy()
)

print("После ограничения повторов одной PERSON:", len(df_sel))

In [ ]:
df_sel["sentence_len_chars"] = df_sel["sentence"].apply(len)
df_sel["sentence_len_words"] = df_sel["sentence"].apply(lambda x: len(str(x).split()))

In [ ]:
def length_bucket(n):
    if n < 70:
        return "short"
    elif n < 130:
        return "medium"
    return "long"

df_sel["length_bucket"] = df_sel["sentence_len_chars"].apply(length_bucket)

print(df_sel["length_bucket"].value_counts())

In [ ]:
format_quota = {
    "two_words": 100,
    "three_plus_words": 120,
    "initials_surname": 120
}

In [ ]:
sample_parts = []

for fmt, fmt_total in format_quota.items():
    part = df_sel[df_sel["person_format"] == fmt].copy()

    length_quota = {
        "short": fmt_total // 3,
        "medium": fmt_total // 3,
        "long": fmt_total - 2 * (fmt_total // 3)
    }

    for lb, lb_take in length_quota.items():
        chunk = part[part["length_bucket"] == lb].head(lb_take)
        sample_parts.append(chunk)

In [ ]:
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.drop_duplicates(subset=["sentence", "person"]).copy()

print(sample_df["person_format"].value_counts())
print(sample_df["length_bucket"].value_counts())

In [ ]:
print("Распределение по форматам:")
print(sample_df["person_format"].value_counts())
print()

print("Распределение по длине:")
print(sample_df["length_bucket"].value_counts())
print()

print(sample_df[["sentence", "person", "person_format", "length_bucket"]].head(20))

In [ ]:
sample_unique_sent = sample_df.drop_duplicates(subset=["sentence"]).copy()

print("Уникальных предложений:", len(sample_unique_sent))

In [ ]:
sample_unique_sent.to_csv("factrueval_person_sample_200.csv", index=False, encoding="utf-8-sig")

In [ ]:
from google.colab import files

files.download("factrueval_person_sample_200.csv")

**Слияние 6 источников в 1 датасет и отбор 1400 чистых текстов с PERSON**

In [ ]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path
from typing import Any

import pandas as pd


In [ ]:
DATA_DIR = Path(".")
OUT_CSV = DATA_DIR / "person_master_dataset.csv"
OUT_XLSX = DATA_DIR / "person_master_dataset.xlsx"


In [ ]:
def safe_literal_eval(value: Any) -> Any:
    if pd.isna(value):
        return None
    if not isinstance(value, str):
        return value
    value = value.strip()
    if not value:
        return None
    try:
        return ast.literal_eval(value)
    except Exception:
        return value


In [ ]:
def count_words(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\S+", text))


In [ ]:
def to_json_str(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False)

In [ ]:
def normalize_common_columns(
    df: pd.DataFrame,
    source_name: str,
    sentence_col: str = "sentence",
    persons_col: str | None = None,
    main_person_col: str | None = None,
    person_norm_col: str | None = None,
    person_format_col: str | None = None,
    n_persons_col: str | None = None,
    n_words_col: str | None = None,
    length_chars_col: str | None = None,
    extra_keep: list[str] | None = None,
) -> pd.DataFrame:
    extra_keep = extra_keep or []

    result = pd.DataFrame()
    result["source"] = source_name
    result["sentence"] = df[sentence_col].astype(str).str.strip()

    if persons_col and persons_col in df.columns:
        result["persons_raw"] = df[persons_col].apply(safe_literal_eval)
    else:
        result["persons_raw"] = None

    if main_person_col and main_person_col in df.columns:
        result["main_person"] = df[main_person_col]
    else:
        result["main_person"] = None

    if person_norm_col and person_norm_col in df.columns:
        result["person_norm"] = df[person_norm_col]
    else:
        result["person_norm"] = None

    if person_format_col and person_format_col in df.columns:
        result["person_format"] = df[person_format_col]
    else:
        result["person_format"] = None

    if n_persons_col and n_persons_col in df.columns:
        result["n_persons"] = df[n_persons_col]
    else:
        result["n_persons"] = result["persons_raw"].apply(
            lambda x: len(x) if isinstance(x, list) else None
        )

    if n_words_col and n_words_col in df.columns:
        result["n_words"] = df[n_words_col]
    else:
        result["n_words"] = result["sentence"].apply(count_words)

    if length_chars_col and length_chars_col in df.columns:
        result["length_chars"] = df[length_chars_col]
    else:
        result["length_chars"] = result["sentence"].str.len()

    meta_cols = [
        c for c in extra_keep
        if c in df.columns and c not in {sentence_col, persons_col, main_person_col,
                                        person_norm_col, person_format_col,
                                        n_persons_col, n_words_col, length_chars_col}
    ]
    result["meta_json"] = df[meta_cols].apply(
        lambda row: to_json_str(row.dropna().to_dict()), axis=1
    ) if meta_cols else "{}"

    return result

In [ ]:
def parse_nerus_conll(path: Path) -> pd.DataFrame:

    sentences = []
    tokens = []
    tags = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line.strip():
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
                continue

            parts = line.rsplit(" ", 1)
            if len(parts) != 2:
                continue

            token, tag = parts
            tokens.append(token)
            tags.append(tag)

    if tokens:
        sentences.append((tokens, tags))

    records = []
    for tokens, tags in sentences:
        sentence = " ".join(tokens).strip()

        persons = []
        current_person = []

        for token, tag in zip(tokens, tags):
            if tag == "B-PERSON":
                if current_person:
                    persons.append(" ".join(current_person))
                current_person = [token]
            elif tag == "I-PERSON":
                if current_person:
                    current_person.append(token)
                else:
                    current_person = [token]
            else:
                if current_person:
                    persons.append(" ".join(current_person))
                    current_person = []

        if current_person:
            persons.append(" ".join(current_person))

        records.append(
            {
                "source": "nerus",
                "sentence": sentence,
                "persons_raw": persons,
                "main_person": persons[0] if persons else None,
                "person_norm": persons[0].lower() if persons else None,
                "person_format": None,
                "n_persons": len(persons),
                "n_words": len(tokens),
                "length_chars": len(sentence),
                "meta_json": to_json_str({"tags": tags}),
            }
        )

    return pd.DataFrame(records)

In [ ]:
def main() -> None:
    fact = pd.read_csv(DATA_DIR / "/content/factrueval_person_sample_200.csv")
    fact_df = normalize_common_columns(
        fact,
        source_name="factrueval2016",
        sentence_col="sentence",
        persons_col="person",
        main_person_col="person",
        person_norm_col="person_norm",
        person_format_col="person_format",
        n_words_col="sentence_len_words",
        length_chars_col="sentence_len_chars",
        extra_keep=["split", "n_words_in_sentence", "n_words_in_person", "length_bucket"],
    )
    fact_df["source"] = "factrueval2016"
    fact_df["persons_raw"] = fact_df["persons_raw"].apply(
        lambda x: [x] if isinstance(x, str) and x else x
    )

    mok = pd.read_csv(DATA_DIR / "/content/mokoron_person_sample_200.csv")
    mok_df = normalize_common_columns(
        mok,
        source_name="mokoron",
        sentence_col="sentence",
        persons_col="persons",
        main_person_col="main_person",
        person_norm_col="person_norm",
        person_format_col="person_format",
        n_persons_col="n_persons",
        n_words_col="n_words",
        length_chars_col="length_chars",
        extra_keep=["length_bucket"],
    )
    mok_df["source"] = "mokoron"

    nerel = pd.read_csv(DATA_DIR / "/content/nerel_person_sample_200.csv")
    nerel_df = normalize_common_columns(
        nerel,
        source_name="nerel",
        sentence_col="sentence",
        persons_col="persons",
        main_person_col="main_person",
        person_norm_col="person_norm",
        person_format_col="person_format",
        n_persons_col="n_persons",
        n_words_col="n_words",
        length_chars_col="length_chars",
        extra_keep=["source_txt", "length_bucket"],
    )
    nerel_df["source"] = "nerel"

    wiki = pd.read_csv(DATA_DIR / "/content/wiki_person_sample_500.csv")
    wiki_df = normalize_common_columns(
        wiki,
        source_name="wikipedia",
        sentence_col="sentence",
        persons_col="good_per_entities",
        main_person_col="main_person",
        person_norm_col="person_norm",
        person_format_col="person_format",
        n_persons_col="n_good_per",
        n_words_col="n_words",
        length_chars_col="sent_len",
        extra_keep=["title", "entities", "per_entities", "n_per", "title_count",
                    "length_bucket", "per_count_bucket"],
    )
    wiki_df["source"] = "wikipedia"

    def wiki_extract_persons(x: Any) -> Any:
        x = safe_literal_eval(x)
        if isinstance(x, list):
            out = []
            for item in x:
                if isinstance(item, dict) and "text" in item:
                    out.append(item["text"])
                else:
                    out.append(str(item))
            return out
        return x

    wiki_df["persons_raw"] = wiki["good_per_entities"].apply(wiki_extract_persons)

    winer = pd.read_csv(DATA_DIR / "/content/winer_person_sample_200_no_wiki_overlap.csv")
    winer_df = normalize_common_columns(
        winer,
        source_name="winer",
        sentence_col="sentence",
        persons_col="persons",
        main_person_col="main_person",
        person_norm_col="person_norm",
        person_format_col="person_format",
        n_persons_col="n_persons",
        n_words_col="n_words",
        length_chars_col="length_chars",
        extra_keep=["tokens", "tags", "sentence_norm", "length_bucket"],
    )
    winer_df["source"] = "winer"

    nerus_df = parse_nerus_conll(DATA_DIR / "/content/person_dataset_nerus_5000_texts.txt")
    nerus_df["source"] = "nerus"

    master = pd.concat(
        [fact_df, mok_df, nerel_df, wiki_df, winer_df, nerus_df],
        ignore_index=True
    )

    master["sentence"] = master["sentence"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    master = master[master["sentence"].str.len() > 0].copy()

    master["persons_raw"] = master["persons_raw"].apply(
        lambda x: to_json_str(x) if isinstance(x, (list, dict)) else (x if pd.notna(x) else "")
    )

    cols = [
        "source",
        "sentence",
        "persons_raw",
        "main_person",
        "person_norm",
        "person_format",
        "n_persons",
        "n_words",
        "length_chars",
        "meta_json",
    ]
    master = master[cols]

    cols_to_drop = [
        "n_persons",
        "n_words",
        "length_chars",
        "meta_json",
        "person_format",
        "main_person"
    ]

    master = master.drop(columns=[c for c in cols_to_drop if c in master.columns])

    master.to_csv("person_master_dataset_clean.csv", index=False)


if __name__ == "__main__":
    main()

In [ ]:
from google.colab import files
files.download("person_master_dataset_clean.csv")

**Генерация PERSON**

In [ ]:
official_templates = [
    "гражданин {first} {last} подал заявление",
    "гражданка {first} {last} обратилась в суд",
    "заявитель {first} {last} направил жалобу",
    "подсудимый {first} {last} был задержан",
    "обвиняемый {first} {last} дал показания",
    "истец {first} {last} требует компенсацию",
    "ответчик {first} {last} не явился в суд",
    "свидетель {first} {last} подтвердил факт",
    "физлицо {first} {last} зарегистрировало компанию",
    "ИП {first} {last} подал документы"
]

In [ ]:
family_templates = [
    "сын {first} {last} сообщил о происшествии",
    "дочь {first} {last} выступила на мероприятии",
    "мать {first} {last} рассказала журналистам",
    "отец {first} {last} дал комментарий",
    "брат {first} {last} присутствовал на встрече",
    "сестра {first} {last} приняла участие в обсуждении",
    "супруг {first} {last} подтвердил информацию",
    "супруга {first} {last} сделала заявление",
    "дедушка {first} {last} поделился воспоминаниями",
    "опекун {first} {last} обратился в органы"
]

In [ ]:
eastern_name_templates = [
    "{first} {father} оглы выступил на встрече",
    "{first} {father} кызы сделала заявление",
    "{first} ибн {father} прибыл в город",
    "По словам {first} {father} оглы, ситуация стабильна",
    "{first} ибн {father} представил доклад",
    "{first} {father} кызы участвовала в конференции",
    "Докладчик {first} {father} оглы отметил важность проекта",
    "{first} ибн {father} прокомментировал новость",
    "{first} {father} оглы подписал документ",
    "{first} {father} кызы выступила с инициативой"
]

In [ ]:
double_surname_templates = [
    "{first} {last}-{last2} выступил на конференции",
    "По словам {first} {last}-{last2}, ситуация изменилась",
    "{first} {last}-{last2} дал интервью журналистам",
    "Эксперт {first} {last}-{last2} прокомментировал события",
    "{first} {last}-{last2} принял участие в заседании",
    "Доклад представил {first} {last}-{last2}",
    "Как отметил {first} {last}-{last2}, решение было сложным",
    "{first} {last}-{last2} сообщил о новых мерах",
    "Встречу посетил {first} {last}-{last2}",
    "{first} {last}-{last2} опубликовал отчет"
]

In [ ]:
!pip install faker

In [ ]:
from faker import Faker
import pandas as pd
import random

fake = Faker("ru_RU")
random.seed(42)

double_surname_templates = [
    "{first} {last}-{last2} выступил на конференции",
    "По словам {first} {last}-{last2}, ситуация изменилась",
    "{first} {last}-{last2} дал интервью журналистам",
    "Эксперт {first} {last}-{last2} прокомментировал события",
    "{first} {last}-{last2} принял участие в заседании",
    "Доклад представил {first} {last}-{last2}",
    "Как отметил {first} {last}-{last2}, решение было сложным",
    "{first} {last}-{last2} сообщил о новых мерах",
    "Встречу посетил {first} {last}-{last2}",
    "{first} {last}-{last2} опубликовал отчет",
]

def generate_name_parts():
    return {
        "first": fake.first_name(),
        "last": fake.last_name(),
        "last2": fake.last_name(),
    }

examples = []

for template in double_surname_templates:
    for _ in range(9):
        parts = generate_name_parts()
        text = template.format(**parts)
        examples.append(text)

random.shuffle(examples)

df = pd.DataFrame({
    "text": examples,
    "format_id": 4,
    "template_group": "double_surname"
})

print(df.head(90))
print(f"Всего примеров: {len(df)}")


In [ ]:
df.to_csv("double_surname_90_all_templates.csv", index=False, encoding="utf-8")

In [ ]:
import pandas as pd
import random

random.seed(42)

first_names = [
    "Ахмед", "Мухаммед", "Али", "Омар", "Хасан",
    "Хусейн", "Ибрагим", "Юсуф", "Абдулла", "Рашид",
    "Тимур", "Азиз", "Карим", "Исмаил", "Саид"
]

father_names = [
    "Ахмед", "Мухаммед", "Али", "Омар", "Хасан",
    "Хусейн", "Ибрагим", "Юсуф", "Абдулла", "Рашид"
]

eastern_name_templates = [
    "{first} {father} оглы выступил на встрече",
    "{first} {father} кызы сделала заявление",
    "{first} ибн {father} прибыл в город",
    "По словам {first} {father} оглы, ситуация стабильна",
    "{first} ибн {father} представил доклад",
    "{first} {father} кызы участвовала в конференции",
    "Докладчик {first} {father} оглы отметил важность проекта",
    "{first} ибн {father} прокомментировал новость",
    "{first} {father} оглы подписал документ",
    "{first} {father} кызы выступила с инициативой"
]

records = []

for i, template in enumerate(eastern_name_templates, start=1):
    for _ in range(5):
        first = random.choice(first_names)
        father = random.choice(father_names)

        text = template.format(first=first, father=father)

        records.append({
            "text": text,
            "format_id": 5,
            "template_id": i,
            "template_text": template
        })

random.shuffle(records)

df = pd.DataFrame(records)

print(df.head(50))
print(f"Всего: {len(df)}")



In [ ]:

df.to_csv("eastern_names_50.csv", index=False, encoding="utf-8")

In [ ]:
from faker import Faker
import pandas as pd
import random

fake = Faker("ru_RU")
random.seed(42)

family_words = [
    "сын", "дочь", "мать", "отец", "брат", "сестра",
    "супруг", "супруга", "дедушка", "бабушка",
    "опекун", "родитель"
]

family_templates = [
    "{family} {person} сообщил о происшествии",
    "{family} {person} выступил на встрече",
    "{family} {person} дал комментарий журналистам",
    "{family} {person} присутствовал на мероприятии",
    "{family} {person} подтвердил информацию",
    "{family} {person} обратился в суд",
    "{family} {person} рассказал о ситуации",
    "{family} {person} принял участие в обсуждении",
    "{family} {person} сделал заявление",
    "{family} {person} выступил с инициативой"
]

def generate_person():
    return f"{fake.first_name()} {fake.last_name()}"

def generate_initial_person():
    return f"{fake.first_name()[0]}.{fake.first_name()[0]}. {fake.last_name()}"

def generate_single_name():
    return fake.last_name()

extra_person_generators = [
    generate_person,
    generate_initial_person,
    generate_single_name
]

records = []

for i in range(100):
    family = random.choice(family_words)
    person = generate_person()

    template = random.choice(family_templates)

    base_text = template.format(family=family, person=person)

    if i < 66:
        n_extra = random.choice([1, 2])

        extras = []
        for _ in range(n_extra):
            gen = random.choice(extra_person_generators)
            extras.append(gen())

        extra_text = ", а также " + " и ".join(extras)

        text = base_text + extra_text + " приняли участие в обсуждении вопроса."
    else:
        text = base_text + " в ходе недавнего события."

    records.append({
        "text": text,
        "format_id": 9,
        "n_entities_estimate": 1 + (2 if i < 66 else 0)
    })

df = pd.DataFrame(records)

print(df.head(50))
print(f"Всего: {len(df)}")



In [ ]:
df.to_csv("family_format_100.csv", index=False, encoding="utf-8")

In [ ]:
from faker import Faker
import pandas as pd
import random

fake = Faker("ru_RU")
random.seed(42)

official_templates = [
    "гражданин {person} подал заявление в районный суд",
    "гражданка {person} обратилась в уполномоченный орган",
    "заявитель {person} направил жалобу в ведомство",
    "подсудимый {person} был доставлен на заседание суда",
    "обвиняемый {person} дал показания следователю",
    "истец {person} требует компенсацию материального ущерба",
    "ответчик {person} не явился на судебное заседание",
    "свидетель {person} подтвердил обстоятельства дела",
    "физлицо {person} зарегистрировало новый договор аренды",
    "ИП {person} подал документы на регистрацию изменений"
]

def gen_full_name():
    return f"{fake.first_name()} {fake.last_name()}"

def gen_initials_plus_surname():
    return f"{fake.first_name()[0]}.{fake.first_name()[0]}. {fake.last_name()}"

def gen_single_name_or_surname():
    return random.choice([fake.first_name(), fake.last_name()])

def gen_position_plus_name():
    positions = [
        "директор", "министр", "руководитель", "эксперт",
        "представитель", "секретарь", "заместитель"
    ]
    return f"{random.choice(positions)} {fake.first_name()} {fake.last_name()}"

extra_person_generators = [
    gen_full_name,
    gen_initials_plus_surname,
    gen_single_name_or_surname,
    gen_position_plus_name
]

context_tails = [
    "в рамках административного разбирательства.",
    "в ходе рассмотрения материалов дела.",
    "после поступления официального уведомления.",
    "в присутствии представителей ведомства.",
    "в соответствии с установленной процедурой.",
    "для дальнейшего рассмотрения обращения.",
    "по итогам проверки представленных документов.",
    "в рамках служебной проверки.",
    "после регистрации обращения в канцелярии.",
    "в рамках установленного порядка."
]

records = []

for template_id, template in enumerate(official_templates, start=1):
    for _ in range(10):
        base_person = gen_full_name()
        base_text = template.format(person=base_person)

        add_extras = random.random() < (2 / 3)

        if add_extras:
            n_extra = random.choice([1, 2])
            extras = [random.choice(extra_person_generators)() for _ in range(n_extra)]

            if n_extra == 1:
                extra_part = f" Дополнительные пояснения представил {extras[0]}."
            else:
                extra_part = f" Дополнительные пояснения представили {extras[0]} и {extras[1]}."

            text = base_text + " " + extra_part + " " + random.choice(context_tails)
            n_entities_estimate = 1 + n_extra
        else:
            text = base_text + " " + random.choice(context_tails)
            n_entities_estimate = 1

        records.append({
            "text": text,
            "format_id": 11,
            "template_id": template_id,
            "template_text": template,
            "n_entities_estimate": n_entities_estimate
        })

random.shuffle(records)

df = pd.DataFrame(records)

print(df.head(10))
print(df["n_entities_estimate"].value_counts().sort_index())



In [ ]:
df.to_csv("official_format_100.csv", index=False, encoding="utf-8")

In [ ]:
import pandas as pd
from pathlib import Path

base_path = Path("/content/person_master_dataset_clean.csv")

synthetic_files = [
    "/content/double_surname_90_all_templates.csv",
    "/content/eastern_names_50.csv",
    "/content/family_format_100.csv",
    "/content/official_format_100.csv",
]

base_df = pd.read_csv(base_path)

print("Основной датасет:")
print(base_df.head())
print(base_df.columns.tolist())
print(len(base_df))

synthetic_dfs = []

for file in synthetic_files:
    df = pd.read_csv(file)


    df = df.copy()
    df["sentence"] = df["text"]
    df["source"] = "synthetic"

    for col in base_df.columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[base_df.columns]
    synthetic_dfs.append(df)

full_df = pd.concat([base_df] + synthetic_dfs, ignore_index=True)

if "sentence" in full_df.columns:
    full_df = full_df.drop_duplicates(subset=["sentence"]).reset_index(drop=True)

print(full_df.head())
print("Строк после объединения:", len(full_df))
print(full_df["source"].value_counts(dropna=False))

output_path = "/content/person_master_dataset_with_synthetic.csv"
full_df.to_csv(output_path, index=False, encoding="utf-8")


In [ ]:
from google.colab import files
files.download("/content/person_master_dataset_with_synthetic.csv")

**Очистка финального датасета**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/person_master_dataset_with_synthetic - person_master_dataset_with_synthetic.csv (1).csv")

df_clean = df.dropna(how='all')

df_clean = df_clean[~df_clean.apply(lambda row: row.astype(str).str.strip().eq('').all(), axis=1)]


In [ ]:
df_clean.to_csv("df_clean.csv", index=False, encoding="utf-8")

from google.colab import files
files.download("df_clean.csv")

Разметка синтетики

In [ ]:
import pandas as pd
import re
import json
from typing import List, Tuple

INPUT_FILE = "/content/synthetic_person_texts - Лист1 (1).csv"
OUTPUT_FILE = "synthetic_person_texts_marked_fixed.csv"

df = pd.read_csv(INPUT_FILE)

possible_text_cols = ["sentence", "text", "TEXT", "Text"]
TEXT_COL = None
for col in possible_text_cols:
    if col in df.columns:
        TEXT_COL = col
        break

NON_PERSON_PREFIXES = [
    "гражданин", "гражданка", "заявитель", "подсудимый", "подсудимая",
    "обвиняемый", "обвиняемая", "истец", "ответчик", "свидетель",
    "физлицо", "ип",

    "сын", "дочь", "мать", "отец", "брат", "сестра",
    "супруг", "супруга", "дедушка", "бабушка", "опекун", "родитель",

    "директор", "министр", "руководитель", "эксперт",
    "представитель", "секретарь", "заместитель", "докладчик",
    "профессор", "ученый", "методист"
]

NON_PERSON_PREFIXES = sorted(set(x.lower() for x in NON_PERSON_PREFIXES), key=len, reverse=True)
NON_PERSON_PREFIXES_SET = set(NON_PERSON_PREFIXES)

BAD_CAPITALIZED_WORDS = {
    "Как", "По", "Встречу", "Доклад", "Дополнительные",
    "Пояснения", "Слова", "Итоги", "Материалы", "Проверки",
    "Процедурой", "Уведомления", "Канцелярии"
}
BAD_CAPITALIZED_WORDS_LOWER = {x.lower() for x in BAD_CAPITALIZED_WORDS}

NAME_WORD = r"[А-ЯЁA-Z][а-яёa-z]+(?:-[А-ЯЁA-Z][а-яёa-z]+)?"

INITIAL_1 = r"[А-ЯЁA-Z]\.?"
INITIALS_2 = rf"{INITIAL_1}\s*{INITIAL_1}"

PATTERNS = [
    ("eastern_ogly_kyzy", re.compile(rf"\b({NAME_WORD}\s+{NAME_WORD}\s+(?:оглы|кызы))\b", flags=re.IGNORECASE)),
    ("eastern_ibn_ben", re.compile(rf"\b({NAME_WORD}\s+(?:ибн|бен)\s+{NAME_WORD})\b", flags=re.IGNORECASE)),
    ("eastern_fon", re.compile(rf"\b({NAME_WORD}\s+фон\s+{NAME_WORD})\b", flags=re.IGNORECASE)),

    ("full_name_3", re.compile(rf"\b({NAME_WORD}\s+{NAME_WORD}\s+{NAME_WORD})\b")),
    ("full_name_2", re.compile(rf"\b({NAME_WORD}\s+{NAME_WORD})\b")),

    ("initials2_before_surname", re.compile(rf"\b({INITIALS_2}\s*{NAME_WORD})\b")),
    ("initial1_before_surname", re.compile(rf"\b({INITIAL_1}\s*{NAME_WORD})\b")),

    ("surname_before_initials2", re.compile(rf"\b({NAME_WORD}\s*{INITIALS_2})\b")),
    ("surname_before_initial1", re.compile(rf"\b({NAME_WORD}\s*{INITIAL_1})\b")),

    ("name_initial_surname", re.compile(rf"\b({NAME_WORD}\s+{INITIAL_1}\s+{NAME_WORD})\b")),

    ("double_surname_only", re.compile(rf"\b({NAME_WORD}-{NAME_WORD})\b")),

    ("single_word", re.compile(rf"\b({NAME_WORD})\b")),
]


def normalize_spaces(text: str) -> str:
    text = str(text)
    text = text.replace("\u00A0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def strip_outer_punct(text: str) -> str:
    return str(text).strip(" \t\n\r,.;:!?()[]{}\"'«»")


def remove_non_person_prefix(candidate: str) -> str:
    c = normalize_spaces(strip_outer_punct(candidate))
    lowered = c.lower()

    for pref in NON_PERSON_PREFIXES:
        if lowered.startswith(pref + " "):
            c = c[len(pref):].strip()
            lowered = c.lower()
            break

    return strip_outer_punct(c)


def is_bad_single_word(word: str) -> bool:
    w = strip_outer_punct(word)
    wl = w.lower()
    if wl in BAD_CAPITALIZED_WORDS_LOWER:
        return True
    if wl in NON_PERSON_PREFIXES_SET:
        return True
    return False


def is_valid_person(candidate: str, pattern_name: str) -> bool:
    c = normalize_spaces(strip_outer_punct(candidate))
    if not c:
        return False

    if c.lower() in NON_PERSON_PREFIXES_SET:
        return False

    if pattern_name == "single_word" and is_bad_single_word(c):
        return False

    if len(c) < 2:
        return False

    if " " not in c and pattern_name == "single_word":
        if is_bad_single_word(c):
            return False

    return True


def fix_candidate(candidate: str) -> str:
    c = normalize_spaces(strip_outer_punct(candidate))
    c = remove_non_person_prefix(c)
    c = strip_outer_punct(c)

    c = re.sub(r"\s*\.\s*", ".", c)
    c = re.sub(r"([А-ЯЁA-Z]\.[А-ЯЁA-Z]\.)([А-ЯЁA-Z][а-яёa-z])", r"\1 \2", c)
    c = re.sub(r"([А-ЯЁA-Z]\.)([А-ЯЁA-Z][а-яёa-z])", r"\1 \2", c)

    return c.strip()


def extract_matches(text: str) -> List[Tuple[int, int, str, str]]:

    text = normalize_spaces(text)
    found = []

    for pattern_name, pattern in PATTERNS:
        for m in pattern.finditer(text):
            start, end = m.span(1)
            candidate = m.group(1)
            candidate = fix_candidate(candidate)

            if not is_valid_person(candidate, pattern_name):
                continue

            found.append((start, end, candidate, pattern_name))

    return found


def choose_longest_non_overlapping(matches: List[Tuple[int, int, str, str]]) -> List[Tuple[int, int, str]]:

    matches_sorted = sorted(
        matches,
        key=lambda x: (-(x[1] - x[0]), x[0], x[1])
    )

    accepted = []
    accepted_spans = []

    for start, end, cand, pattern_name in matches_sorted:
        overlap = False
        for s2, e2, _ in accepted:
            if not (end <= s2 or start >= e2):
                overlap = True
                break
        if not overlap:
            accepted.append((start, end, cand))
            accepted_spans.append((start, end))

    accepted = sorted(accepted, key=lambda x: x[0])
    return accepted


def deduplicate_entities(entities: List[str]) -> List[str]:
    result = []
    seen = set()
    for ent in entities:
        ent_norm = normalize_spaces(ent)
        if ent_norm not in seen:
            result.append(ent_norm)
            seen.add(ent_norm)
    return result


def postprocess_entities(entities: List[str]) -> List[str]:
    clean = []
    for ent in entities:
        e = fix_candidate(ent)

        if not e:
            continue
        if e.lower() in NON_PERSON_PREFIXES_SET:
            continue
        if e.lower() in BAD_CAPITALIZED_WORDS_LOWER:
            continue

        clean.append(e)

    return deduplicate_entities(clean)


def extract_persons(text: str) -> List[str]:
    matches = extract_matches(text)
    best = choose_longest_non_overlapping(matches)
    entities = [cand for _, _, cand in best]
    entities = postprocess_entities(entities)
    return entities


df[TEXT_COL] = df[TEXT_COL].astype(str).apply(normalize_spaces)

df["persons_raw_auto"] = df[TEXT_COL].apply(extract_persons)
df["n_entities_real"] = df["persons_raw_auto"].apply(len)
df["persons_raw_auto_json"] = df["persons_raw_auto"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)

if "persons_raw" in df.columns:
    def is_empty_value(x):
        if pd.isna(x):
            return True
        s = str(x).strip()
        return s == "" or s == "[]" or s.lower() == "nan"

    empty_mask = df["persons_raw"].apply(is_empty_value)
    df.loc[empty_mask, "persons_raw"] = df.loc[empty_mask, "persons_raw_auto_json"]


print(df[[TEXT_COL, "persons_raw_auto", "n_entities_real"]].head(20))

print(df["n_entities_real"].value_counts().sort_index())

problem_zero = df[df["n_entities_real"] == 0]

def has_bad_entity(lst):
    for x in lst:
        if x.lower() in BAD_CAPITALIZED_WORDS_LOWER:
            return True
    return False

problem_bad = df[df["persons_raw_auto"].apply(has_bad_entity)]

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

Дубликаты удалим

In [ ]:
import pandas as pd
import ast

df_clean_1 = pd.read_csv("/content/df_clean - df_clean.csv.csv")



def normalize_persons_raw(value):

    try:
        if pd.isna(value):
            return tuple()

        if isinstance(value, str):
            value = ast.literal_eval(value)

        if not isinstance(value, list):
            return tuple()

        cleaned = []
        for x in value:
            if isinstance(x, str):
                x_norm = " ".join(x.strip().split())
                if x_norm:
                    cleaned.append(x_norm)

        cleaned_unique_sorted = tuple(sorted(set(cleaned)))
        return cleaned_unique_sorted

    except Exception:
        return tuple()

before = len(df_clean_1)

df_clean_1["persons_raw_norm"] = df_clean_1["persons_raw"].apply(normalize_persons_raw)

df_clean_1 = df_clean_1.drop_duplicates(subset=["persons_raw_norm"]).copy()

df_clean_1 = df_clean_1.drop(columns=["persons_raw_norm"])

after = len(df_clean_1)

print(f"Удалено дублей: {before - after}")

In [ ]:
import pandas as pd
import ast
import re

def parse_persons_raw(value):
    try:
        if pd.isna(value):
            return []
        if isinstance(value, list):
            return value
        if isinstance(value, str):
            value = ast.literal_eval(value)
            if isinstance(value, list):
                return value
        return []
    except Exception:
        return []

def has_latin_letters(persons):
    latin_pattern = re.compile(r"[A-Za-z]")
    for p in persons:
        if isinstance(p, str) and latin_pattern.search(p):
            return True
    return False

def has_roman_numerals(persons):
    roman_pattern = re.compile(r"\b[IVXLCDM]+\b", flags=re.IGNORECASE)
    for p in persons:
        if isinstance(p, str) and roman_pattern.search(p):
            return True
    return False

def has_more_than_5_entities(persons):
    return len(persons) > 5

before = len(df_clean_1)

df_clean_1["persons_raw_list"] = df_clean_1["persons_raw"].apply(parse_persons_raw)

latin_before = len(df_clean_1)
df_clean_1 = df_clean_1[~df_clean_1["persons_raw_list"].apply(has_latin_letters)].copy()
latin_removed = latin_before - len(df_clean_1)

roman_before = len(df_clean_1)
df_clean_1 = df_clean_1[~df_clean_1["persons_raw_list"].apply(has_roman_numerals)].copy()
roman_removed = roman_before - len(df_clean_1)

many_before = len(df_clean_1)
df_clean_1 = df_clean_1[~df_clean_1["persons_raw_list"].apply(has_more_than_5_entities)].copy()
many_removed = many_before - len(df_clean_1)

df_clean_1 = df_clean_1.drop(columns=["persons_raw_list"])

after = len(df_clean_1)

print(f"Стало строк: {after}")

Отбор 1400 строк

In [ ]:
df_clean_1.to_csv("df_clean_PERSON_final_4206.csv", index=False, encoding="utf-8")

In [ ]:
from google.colab import files
files.download("df_clean_PERSON_final_4206.csv")

In [ ]:
import pandas as pd
import ast

def parse_persons(value):
    try:
        if pd.isna(value):
            return []
        if isinstance(value, list):
            return value
        if isinstance(value, str):
            return ast.literal_eval(value)
        return []
    except:
        return []

df_clean_1["persons_list"] = df_clean_1["persons_raw"].apply(parse_persons)

df_clean_1["n_entities"] = df_clean_1["persons_list"].apply(len)

df_1 = df_clean_1[df_clean_1["n_entities"] == 1].copy()
df_2 = df_clean_1[df_clean_1["n_entities"] == 2].copy()
df_3 = df_clean_1[df_clean_1["n_entities"] == 3].copy()
df_4_plus = df_clean_1[df_clean_1["n_entities"] >= 4].copy()

print("1 сущность:", len(df_1))
print("2 сущности:", len(df_2))
print("3 сущности:", len(df_3))
print("4+ сущностей:", len(df_4_plus))
print("Всего:", len(df_clean_1))

In [ ]:
df_4_plus.head(63)

In [ ]:
df_4_plus.to_csv("df_4_plus.csv", index=False, encoding="utf-8-sig")

In [ ]:
from google.colab import files
files.download("df_4_plus.csv")

Отбор текстов с 4+ людьми

In [ ]:
import pandas as pd

df_4_plus = pd.read_csv("df_4_plus_marked - df_4_plus.csv.csv")

df_4_plus_clean = df_4_plus[df_4_plus["status"] != "нет"].copy()

df_4_plus_clean = df_4_plus_clean[
    ~(
        (df_4_plus_clean["n_entities"] == 5 ) &
        (df_4_plus_clean["formats"] == "один")
    )
].copy()

print("Осталось строк:", len(df_4_plus_clean))

In [ ]:
df_4_plus_clean.to_csv("df_4_plus_clean.csv", index=False, encoding="utf-8-sig")

Отбор текстов с 3 людьми

In [ ]:
df_3.to_csv("df_3.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("df_3.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("df_3_marked - df_3.csv.csv")

df_clean = df[df["status"] != "нет"].copy()


In [ ]:
df_clean.to_csv("df_3_clean.csv", index=False, encoding="utf-8-sig")

Отбор текстов с 2 людьми

In [ ]:
df_2.to_csv("df_2.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("df_2.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/df_2_clean - df_2.csv.csv")

df_2_clean_no_empty = df.dropna(how="all").copy()

df_2_clean_no_empty = df_2_clean_no_empty[
    ~df_2_clean_no_empty.apply(
        lambda row: all(str(x).strip() == "" or pd.isna(x) for x in row),
        axis=1
    )
].copy()

df_2_clean_no_empty.to_csv("df_2_clean_no_empty.csv", index=False, encoding="utf-8-sig")


In [ ]:
df_2_clean_no_empty.to_csv("df_2_clean_no_empty.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("df_2_clean_no_empty.csv")

Отбор текстов с 1 PERSON

In [ ]:
df_1

In [ ]:
df_1.to_csv("df_1.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("df_1.csv")

100 с инициалами:

In [ ]:
import pandas as pd

df = pd.read_csv('/content/df_1_без_синтетики - df_1.csv.csv')

def count_dots(x):
    return str(x).count('.')

df_1_dot = df[df['persons_raw'].apply(count_dots) == 1]

df_2_dots = df[df['persons_raw'].apply(count_dots) == 2]

df_3_dots = df[df['persons_raw'].apply(count_dots) == 3]


In [ ]:

def extract_person(x):
    try:
        val = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(val, list) and len(val) > 0:
            return str(val[0])
        return str(val)
    except:
        return str(x)

def count_dots_person(x):
    return extract_person(x).count('.')

df_1_dot = df[df['persons_raw'].apply(count_dots_person) == 1].copy()
df_2_dots = df[df['persons_raw'].apply(count_dots_person) == 2].copy()

sample_1_dot = df_1_dot.sample(n=11, random_state=42)

df_selected = pd.concat([df_2_dots, sample_1_dot], ignore_index=True)

df_selected = df_selected.sample(frac=1, random_state=42).reset_index(drop=True)

output_file = 'df_2dots_plus_11_from_1dot.xlsx'
df_selected.to_excel(output_file, index=False)




Отбор 2 слова

In [ ]:
import pandas as pd
import ast

df = pd.read_csv('/content/df_1_без_синтетики_marked - df_1.csv (1).csv')

def extract_person(x):
    try:
        val = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(val, list) and len(val) > 0:
            return str(val[0])
        return ""
    except:
        return ""

def count_words(x):
    person = extract_person(x)
    return len(person.strip().split())

df_filtered = df[
    (df['persons_raw'].apply(count_words) == 2) &
    (df['status'].isna() | (df['status'] == ""))
].copy()

print(len(df_filtered))

df_filtered.to_csv('df_2words_empty_status.csv', index=False)

Отбор 3 слова - ФИО

In [ ]:
import pandas as pd
import ast

df = pd.read_csv('/content/df_1_без_синтетики_marked - df_1.csv (1).csv')

def extract_person(x):
    try:
        val = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(val, list) and len(val) > 0:
            return str(val[0])
        return ""
    except:
        return ""

def is_three_capitalized_words(x):
    person = extract_person(x).strip()
    words = person.split()

    if len(words) != 3:
        return False

    return all(w[0].isupper() for w in words if len(w) > 0)

df_filtered = df[
    df['persons_raw'].apply(is_three_capitalized_words) &
    (df['status'].isna() | (df['status'] == ""))
].copy()

df_filtered.to_csv('df_3words_capitalized_empty_status.csv', index=False)

Отбор вручную и загрузка 1 слово и должность + человек

In [ ]:
import pandas as pd

df = pd.read_csv('/content/df_1_без_синтетики_marked - df_1.csv (1).csv')

df_filtered = df[
    (df['status'] == 1) |
    (df['status'].astype(str).str.lower() == 'ок')
].copy()




In [ ]:
df_filtered.to_excel('df_status_1_or_ok.xlsx', index=False)